**Create Table on BigQuery**

In [7]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

# =========================================
# CONFIGURATION
# =========================================

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Scans_Ref"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# =========================================
# INITIALIZE BIGQUERY CLIENT
# =========================================

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# TABLE SCHEMA
# =========================================

schema = [
    # Primary Key
    bigquery.SchemaField(
        "Scan_Code",
        "STRING",
        mode="REQUIRED"
    ),

    # Scan Information
    bigquery.SchemaField(
        "Scan_Group",
        "STRING"
    ),

    bigquery.SchemaField(
        "Scan_Name",
        "STRING",
        mode="REQUIRED"
    )
]

# =========================================
# CREATE TABLE OBJECT
# =========================================

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

# =========================================
# CLUSTERING
# =========================================

# Clustering by scan group and scan code
table.clustering_fields = [
    "Scan_Group",
    "Scan_Code"
]

# =========================================
# CREATE TABLE
# =========================================

table = client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("Table created successfully")
print(TABLE_REF)
print("=================================")

Table created successfully
depihealthnux.Depihealthnux_Main.Scans_Ref


**Insert Data into BigQuery**

In [8]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# GOOGLE BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

# =========================================
# CONFIGURATION
# =========================================

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Scans_Ref"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# =========================================
# INITIALIZE BIGQUERY CLIENT
# =========================================

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# READ EXCEL FILE
# =========================================

df = pd.read_excel("../Resources/Scans_Ref.xlsx")

# =========================================
# CLEAN DATA
# =========================================

df = df.fillna("")

df["Scan_Code"] = df["Scan_Code"].astype(str).str.strip()
df["Scan_Group"] = df["Scan_Group"].astype(str).str.strip()
df["Scan_Name"] = df["Scan_Name"].astype(str).str.strip()

# =========================================
# LOAD TO BIGQUERY
# =========================================

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"  # Replace all existing data
)

job = client.load_table_from_dataframe(
    df,
    TABLE_REF,
    job_config=job_config
)

job.result()

print("=================================")
print(f"{len(df)} rows loaded successfully")
print(TABLE_REF)
print("=================================")

8 rows loaded successfully
depihealthnux.Depihealthnux_Main.Scans_Ref


**Create Postgres Table**

In [9]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT TO POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS scan_no_seq;

""")

# =========================================
# CREATE TABLE QUERY
# =========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS Scans_Ref (

    -- Auto Generated Primary Key
    Scan_Code TEXT PRIMARY KEY
    DEFAULT (
        'SC' ||
        LPAD(
            nextval('scan_no_seq')::TEXT,
            3,
            '0'
        )
    ),

    -- Scan Information
    Scan_Group TEXT,
    Scan_Name TEXT NOT NULL

);
"""

# =========================================
# EXECUTE QUERY
# =========================================

cursor.execute(create_table_query)

# =========================================
# COMMIT CHANGES
# =========================================

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Scans_Ref")
print("=================================")

# =========================================
# CLOSE CONNECTION
# =========================================

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Scans_Ref


**Test BigQuery to Postgres Sync**

In [10]:
import sys
import pandas as pd
import psycopg2

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ FROM BIGQUERY
# =========================================

query = """
SELECT
    Scan_Code,
    Scan_Group,
    Scan_Name
FROM `depihealthnux.Depihealthnux_Main.Scans_Ref`
ORDER BY Scan_Code
"""

df = client.query(query).to_dataframe()

print("\n=================================")
print("DATA RECEIVED FROM BIGQUERY")
print("=================================")
print(f"Rows Retrieved: {len(df)}")

print("\nFirst 3 Rows From BigQuery:")
print(df.head(3))

# Uncomment if you want the full dataframe
# print(df)

# =========================================
# CONNECT TO POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CLEAR EXISTING DATA
# =========================================

cursor.execute("""
DELETE FROM Scans_Ref;
""")

# =========================================
# INSERT DATA
# =========================================

if not df.empty:

    execute_values(
        cursor,
        """
        INSERT INTO Scans_Ref (
            Scan_Code,
            Scan_Group,
            Scan_Name
        )
        VALUES %s
        """,
        df.values.tolist()
    )

# =========================================
# RESET SEQUENCE
# =========================================

cursor.execute("""

SELECT setval(
    'scan_no_seq',
    COALESCE(
        (
            SELECT MAX(
                CAST(
                    REPLACE(Scan_Code, 'SC', '')
                    AS INTEGER
                )
            )
            FROM Scans_Ref
        ),
        1
    ),
    true
);

""")

# =========================================
# COMMIT CHANGES
# =========================================

conn.commit()

print("\n=================================")
print(f"Inserted {len(df)} rows into PostgreSQL")
print("=================================")

# =========================================
# VERIFY INSERTION
# =========================================

cursor.execute("""

SELECT
    Scan_Code,
    Scan_Group,
    Scan_Name
FROM Scans_Ref
ORDER BY Scan_Code
LIMIT 3

""")

rows = cursor.fetchall()

print("\nFirst 3 Rows From PostgreSQL:")

for row in rows:
    print(row)

# =========================================
# CLOSE CONNECTION
# =========================================

cursor.close()
conn.close()

print("\n=================================")
print("SYNC COMPLETED SUCCESSFULLY")
print("=================================")


DATA RECEIVED FROM BIGQUERY
Rows Retrieved: 8

First 3 Rows From BigQuery:
  Scan_Code Scan_Group   Scan_Name
0     SC001         US      US_Abd
1     SC002         US  US_Thyroid
2     SC003         US  US_Cranial

Inserted 8 rows into PostgreSQL

First 3 Rows From PostgreSQL:
('SC001', 'US', 'US_Abd')
('SC002', 'US', 'US_Thyroid')
('SC003', 'US', 'US_Cranial')

SYNC COMPLETED SUCCESSFULLY


**Test Postgres to BigQuery Sync**

In [11]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ FROM POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT
    Scan_Code,
    Scan_Group,
    Scan_Name
FROM Scans_Ref
ORDER BY Scan_Code

"""

df = pd.read_sql(query, conn)

print("\n=================================")
print("DATA RECEIVED FROM POSTGRESQL")
print("=================================")
print(f"Rows: {len(df)}")
print(df)

print("\nFirst 3 Rows:")
print(df.head(3))

conn.close()

# =========================================
# LOAD TO BIGQUERY
# =========================================

TABLE_REF = (
    "depihealthnux."
    "Depihealthnux_Main."
    "Scans_Ref"
)

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    df,
    TABLE_REF,
    job_config=job_config
)

job.result()

print("\n=================================")
print(f"Loaded {len(df)} rows into BigQuery")
print("=================================")

# =========================================
# VERIFY INSERTION
# =========================================

verify_query = """

SELECT
    Scan_Code,
    Scan_Group,
    Scan_Name
FROM `depihealthnux.Depihealthnux_Main.Scans_Ref`
ORDER BY Scan_Code
LIMIT 3

"""

verify_df = client.query(verify_query).to_dataframe()

print("\nFirst 3 Rows From BigQuery:")
print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_16164\980039034.py:43: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)



DATA RECEIVED FROM POSTGRESQL
Rows: 8
  scan_code scan_group   scan_name
0     SC001         US      US_Abd
1     SC002         US  US_Thyroid
2     SC003         US  US_Cranial
3     SC004        MRI   MRI_Chest
4     SC005         US   MRI_Brain
5     SC006         CT    CT_Chest
6     SC007         CT    CT_Brain
7     SC008         CT  CT_Abdomen

First 3 Rows:
  scan_code scan_group   scan_name
0     SC001         US      US_Abd
1     SC002         US  US_Thyroid
2     SC003         US  US_Cranial

Loaded 8 rows into BigQuery

First 3 Rows From BigQuery:
  Scan_Code Scan_Group   Scan_Name
0     SC001         US      US_Abd
1     SC002         US  US_Thyroid
2     SC003         US  US_Cranial
